# Real Estate Crawling: Filtering, Concurrency, Sitemaps & SPAs

This notebook demonstrates the features that turn a broad link-follower into a
focused **document-inventory** crawler for real-estate and municipal data:

1. **Configurable concurrency** — tune throughput with `max_concurrency` and
   the connection-pool limits.
2. **URL filtering rules** (`CrawlRules`) — follow only the URLs you care
   about and skip crawl traps (login pages, sort/calendar links, binaries,
   off-site domains).
3. **Sitemap discovery** — seed the queue from a site's own published
   document inventory before link-following begins.
4. **JavaScript-rendered sites** — swap in a headless-browser fetcher for
   listing portals built as single-page apps, where plain HTTP sees an empty
   shell.
5. **Flushing to disk** — stream a crawl too big for memory to a JSONL file.

We use a municipal property/GIS site whose street pages link to many
`?Letter=` query-parameter variants — a classic crawl trap that filtering
rules are designed to tame — and a public React demo app for the SPA section.

In [ ]:
import logging
import time
from urllib.parse import urlparse, parse_qs

from linktrace import Spider, CrawlRules, Crawler, SitemapParser
from linktrace import RoutingFetcher, BrowserFetcher, looks_like_spa_shell

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

START_URL = "https://gis.vgsi.com/WoonsocketRI/Streets.aspx"
CACHE = ".feature_demo_cache"  # cache so repeated runs are fast

## 1. Configurable concurrency

`max_concurrency` controls how many URLs are fetched per batch (default 10),
while `max_connections` / `max_connections_per_host` size the underlying
aiohttp connection pool. Raise them to go faster across many hosts; keep
`max_connections_per_host` modest to stay polite to a single server.

In [ ]:
spider = Spider(
    start_url=START_URL,
    max_depth=1,
    max_concurrency=20,            # up to 20 fetches per batch (default 10)
    max_connections=100,           # total pool size
    max_connections_per_host=6,    # be gentle on this one host
    cache_dir=CACHE,
    show_progress=True,
)

start = time.time()
documents = await spider.run_async()
elapsed = time.time() - start

print(f"\nCrawled {len(documents)} pages in {elapsed:.1f}s "
      f"(max_concurrency={spider.max_concurrency})")

## 2. URL filtering rules — taming query-param crawl traps

Each street page links to dozens of `?Letter=A`, `?Letter=B`, ... variants.
Without rules those all get queued. Let's first measure how many discovered
links carry a `Letter` parameter.

In [ ]:
discovered = [link.url for doc in documents for link in doc.internal_links]
with_param = [u for u in discovered if 'Letter=' in u]

print(f"Total internal links discovered: {len(discovered)}")
print(f"  ...carrying a Letter= query param: {len(with_param)}")
print("\nExamples:")
for u in with_param[:5]:
    print(f"  {u}")

Now define `CrawlRules` to keep the crawl on-site and drop the parameter
variants, binaries, and any off-domain links. We compare how many links
survive filtering.

In [ ]:
rules = CrawlRules(
    allowed_domains=["gis.vgsi.com"],   # stay on-site (subdomain-aware)
    exclude_query_params=["Letter"],     # drop the ?Letter= crawl trap
    blocked_extensions=["pdf", "jpg", "png", "zip"],
)

kept = [u for u in discovered if rules.allows(u)]
dropped = [u for u in discovered if not rules.allows(u)]

print(f"Links kept by rules:    {len(kept)}")
print(f"Links dropped by rules: {len(dropped)}")
print("\nSample kept:")
for u in kept[:5]:
    print(f"  {u}")

### How rules are evaluated

Rules are checked against every **discovered** link before it is queued (the
`start_url` is always fetched). For each dimension a populated *allow* list
is a whitelist and a *block*/*exclude* list rejects matches — and
**exclusions always win**. The `allows()` method is pure, so it's easy to
unit-test your policy in isolation:

In [ ]:
preset = CrawlRules(
    allowed_domains=["realty.example.com"],
    include_path_prefixes=["/homes-for-sale/", "/agents/"],
    exclude_path_prefixes=["/login", "/privacy"],
    blocked_extensions=["pdf", "jpg"],
    exclude_query_params=["sort", "view", "calendar"],
    exclude_patterns=[r"/print/?$"],
)

checks = [
    "https://realty.example.com/homes-for-sale/123",        # keep
    "https://realty.example.com/agents/jane",               # keep
    "https://realty.example.com/login",                     # exclude prefix
    "https://realty.example.com/homes-for-sale/1?sort=asc", # exclude param
    "https://realty.example.com/homes-for-sale/flyer.pdf",  # exclude ext
    "https://realty.example.com/homes-for-sale/1/print",    # exclude regex
    "https://other-site.com/homes-for-sale/1",              # off-domain
]
for url in checks:
    verdict = "KEEP " if preset.allows(url) else "DROP "
    print(f"  {verdict} {url}")

## 3. Sitemap discovery

Link-following only finds pages reachable from the start URL. Sitemaps expose
a site's full document inventory. `SitemapParser.parse()` is namespace-
agnostic and handles both `<urlset>` listings and `<sitemapindex>` files.

In [ ]:
URLSET = b'''<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">
  <url><loc>https://realty.example.com/homes-for-sale/1</loc></url>
  <url><loc>https://realty.example.com/homes-for-sale/2</loc></url>
</urlset>'''

kind, locs = SitemapParser.parse(URLSET)
print(f"Parsed a {kind} with {len(locs)} URLs:")
for u in locs:
    print(f"  {u}")

Against a live site, `Crawler.discover_sitemap_urls()` reads `Sitemap:`
declarations from `robots.txt` (falling back to `/sitemap.xml`), follows
nested sitemap indexes, and returns the page URLs. It's best-effort — if no
sitemap exists it simply returns an empty list.

In [ ]:
async with Crawler(cache_dir=CACHE) as crawler:
    sitemap_urls = await crawler.discover_sitemap_urls(START_URL)

print(f"Sitemap discovery returned {len(sitemap_urls)} URL(s)")
for u in sitemap_urls[:10]:
    print(f"  {u}")
if not sitemap_urls:
    print("  (this site publishes no sitemap — link-following still works)")

## 4. Putting it together — a focused real-estate preset

Combine all three: seed from sitemaps, filter discovered links, and tune
concurrency. The same `CrawlRules` is applied to sitemap-seeded URLs *and*
links found while crawling.

In [ ]:
focused = Spider(
    start_url=START_URL,
    max_depth=2,
    use_sitemaps=True,                  # seed from sitemap.xml / robots.txt
    rules=CrawlRules(                   # ...and filter everything
        allowed_domains=["gis.vgsi.com"],
        exclude_query_params=["Letter"],
        blocked_extensions=["pdf", "jpg", "png"],
    ),
    max_concurrency=20,
    cache_dir=CACHE,
    show_progress=True,
)

docs = await focused.run_async()
print(f"\nFocused crawl fetched {len(docs)} pages")

# Confirm the Letter= trap was avoided
trap = [d for d in docs if 'Letter=' in d.url]
print(f"Pages with a Letter= param (should be 0): {len(trap)}")

## 5. JavaScript-rendered listings (SPAs)

Many listing portals and GIS front ends are single-page apps: the server
returns a near-empty HTML shell and React/Vue/Angular builds the page — and
every link on it — in the browser. The default HTTP fetcher gets the shell and
finds nothing to follow.

linktrace's transport is pluggable. `BrowserFetcher` renders each page in
headless Chromium via Playwright; `RoutingFetcher` tries plain HTTP first and
escalates to the browser only when the response looks like an un-rendered
shell, so server-rendered pages stay on the cheap path.

This section needs the optional browser extra:

```bash
pip install "linktrace[browser]"
playwright install chromium
```

First, look at what plain HTTP sees on a public React demo app.

In [ ]:
SPA_URL = "https://react-shopping-cart-67954.firebaseapp.com/"

async with Crawler(respect_robots_txt=False) as crawler:
    raw = await crawler.fetcher.fetch(SPA_URL, {"User-Agent": crawler.user_agent})

print(f"HTTP {raw.status}, {len(raw.body or '')} bytes")
print(f"anchors in raw HTML : {(raw.body or '').lower().count('<a ')}")
print(f"looks_like_spa_shell: {looks_like_spa_shell(raw)}")

Zero anchors and the heuristic flags it: a `<noscript>` asking you to enable
JavaScript plus an empty `<div id="root">`. Now crawl the same URL with
`RoutingFetcher`. Everything else — rules, caching, rate limiting, exports —
is unchanged; only the transport differs.

In [ ]:
routing = RoutingFetcher(fallback=BrowserFetcher(max_pages=2))

spa_spider = Spider(
    start_url=SPA_URL,
    max_depth=1,
    fetcher=routing,
    rules=CrawlRules(allowed_domains=["react-shopping-cart-67954.firebaseapp.com"]),
    cache_dir=CACHE,
)

spa_docs = await spa_spider.run_async()

print(f"Crawled {len(spa_docs)} page(s); {routing.escalations} escalated to the browser\n")
for d in spa_docs:
    print(f"[{d.fetcher:7s}] {d.url}")
    print(f"           {len(d.internal_links)} internal / {len(d.external_links)} external links")
    for link in d.external_links[:3]:
        print(f"             -> {link.url}")

The start page was fetched over HTTP, recognised as a shell, re-fetched in the
browser and parsed from the rendered DOM — `doc.fetcher == "browser"` records
that. Pages that render fine over HTTP keep `fetcher == "http"`, and the
`fetcher` column comes through in every export so downstream code can tell
them apart.

Two knobs matter on real portals:

- `render_patterns=[r"/search", r"/map"]` sends matching URLs straight to the
  browser without the HTTP probe.
- `keep_hash_routes=True` on the Spider keeps `/#/listing/42`-style routes as
  distinct URLs instead of collapsing them to the bare page.

## 6. Big crawls: flush to disk

A deep crawl of a listing portal can run to tens of thousands of pages, and
rendered DOMs are several times larger than raw HTML. `flush_to` appends each
page to a JSONL file the moment it is crawled and keeps nothing in memory
beyond URL bookkeeping. The line format is the same as `Serializers.to_json`,
so it loads straight into pandas or polars.

In [ ]:
import json

OUT = "focused_crawl.jsonl"

streamed = Spider(
    start_url=START_URL,
    max_depth=1,
    rules=CrawlRules(allowed_domains=["gis.vgsi.com"], exclude_query_params=["Letter"]),
    cache_dir=CACHE,
    flush_to=OUT,          # one JSON object per line, written as each page completes
    flush_html=False,      # True to keep page source as well
)

result = await streamed.run_async()
print(f"run_async() returned {len(result)} documents (nothing retained in memory)")

with open(OUT) as f:
    rows = [json.loads(line) for line in f]
print(f"{len(rows)} pages in {OUT}")
print(rows[0]["url"], "->", len(rows[0]["internal_links"]), "internal links")

## Takeaways

- **`max_concurrency`** + connection-pool limits tune throughput without
  touching the rest of your config.
- **`CrawlRules`** turns an exhaustive link-follower into a targeted
  document crawler; `allows()` is pure and unit-testable.
- **`use_sitemaps=True`** discovers pages link-following would miss, and the
  same rules keep that inventory on-topic.
- **`fetcher=RoutingFetcher()`** makes JavaScript-rendered portals crawlable
  with one argument, paying the browser cost only where it is needed.
- **`flush_to="crawl.jsonl"`** keeps memory flat on crawls that would never
  fit in RAM.

See the [Examples](https://linktrace.readthedocs.io/examples/) and
[API Reference](https://linktrace.readthedocs.io/api-reference/) docs for the
full parameter list.